# Introduction
This notebook is a clone of the run_quantize.py file for mimicking and debugging experiments without using seml and slurm

# 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
# Setting up environment
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

import seml
import re
import shutil

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
HUB_PATH = "/nfs/students/daro/.cache/huggingface/hub/"

if not os.path.exists(HUB_PATH):
    os.makedirs(HUB_PATH)
    print(f"Creating huggingface hub path at {HUB_PATH}")
    
print(f"Setting cache path to {CACHE_PATH}")
os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
import logging
logger = logging.getLogger("quant_logger")
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 1007.72 GB
MemFree: 816.32 GB
MemAvailable: 982.62 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successfu

# 2. Debug locally quantized models

In [7]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from src import MODEL_SAVE_PATH
from src.data.FKTC_datasets import load_dataset_from_name
from src.reliability.response_generator import ResponseGenerator
import random

# Constants
META_LLAMA_3_8B = "meta-llama/Meta-Llama-3-8B"

# Model configurations
local_quantized_models = {
    "Llama-3-8B-AWQ-4bit-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-AWQ-4"),
    "Llama-3-8B-BNB-4bit-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-BNB-4"),
    "Llama-3-8B-HQQ-mixed-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-HQQ-mixed"),
}

def load_model_and_tokenizer(model_name):
    model_path = local_quantized_models[model_name]
    tokenizer = AutoTokenizer.from_pretrained(META_LLAMA_3_8B)
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")
    return model, tokenizer

def print_model_attributes(model):
    print("Model Attributes:")
    for attr in dir(model):
        if not attr.startswith("_"):
            print(f"{attr}: {getattr(model, attr)}")

def generate_response(generator, query, strategy, dataset_name):
    responses = generator.generate_responses(
        [query],
        strategy,
        dataset_name,
        [None],  # true_answers
        max_new_tokens=25,
        temperature=0.1,
        use_beam_search=False,
        n_repeats=1,
        n_beams=1
    )
    return responses[0][0]['output_text']

# Load model (default to AWQ)
model_name = "Llama-3-8B-AWQ-4bit-local"
print(f"Loading model: {model_name}")
model, tokenizer = load_model_and_tokenizer(model_name)

# Print model attributes
print_model_attributes(model)

# # Initialize ResponseGenerator
# generator = ResponseGenerator(model_name=model_name, model=model, tokenizer=tokenizer)

# # Load a dataset
# dataset_name = "P101"
# dataset = load_dataset_from_name(dataset_name, max_entries=5)

# # Typo types and intensities
# typo_types = [
#     "none", "char_insertion", "char_deletion", "char_replacement", "char_repetition",
#     "char_swapping", "word_CMW", "char_LCC", "word_synonym", "char_insert_noise",
#     "word_repeat", "char_substitution", "word_emoji", "word_internet_slang",
#     "word_phrase_translation", "word_context_aware_insertion", "word_remove_punctuation",
#     "word_keyword_only", "word_taxonomy_pos", "word_taxonomy_neg"
# ]
# intensities = [1, 2, 3]

# # Generate responses
# for i, (query, _) in enumerate(dataset):
#     print(f"\nOriginal Query {i+1}: {query}")
    
#     for typo_type in typo_types:
#         for intensity in intensities:
#             # Apply typo modification
#             modified_dataset = load_dataset_from_name(
#                 dataset_name,
#                 max_entries=1,
#                 typo_type=typo_type,
#                 typo_intensity=intensity
#             )
#             modified_query = modified_dataset[0][0]
            
#             print(f"  Typo: {typo_type}, Intensity: {intensity}")
#             print(f"  Modified Query: {modified_query}")
            
#             # Generate response
#             response = generate_response(generator, modified_query, "Direct Completion", dataset_name)
#             print(f"  Response: {response}")
#             print("-" * 50)

Loading model: Llama-3-8B-AWQ-4bit-local


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model Attributes:
T_destination: ~T_destination
active_adapter: <bound method PeftAdapterMixin.active_adapter of LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): WQLinear_GEMM(in_features=4096, out_features=4096, bias=False, w_bit=4, group_size=128)
          (k_proj): WQLinear_GEMM(in_features=4096, out_features=1024, bias=False, w_bit=4, group_size=128)
          (v_proj): WQLinear_GEMM(in_features=4096, out_features=1024, bias=False, w_bit=4, group_size=128)
          (o_proj): WQLinear_GEMM(in_features=4096, out_features=4096, bias=False, w_bit=4, group_size=128)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): WQLinear_GEMM(in_features=4096, out_features=14336, bias=False, w_bit=4, group_size=128)
          (up_proj): WQLinear_GEMM(in_features=4096, out_features=14336

In [11]:
model.config.quantization_config.bits

4

# 3. Bits

In [5]:
import os
import torch
from transformers import AutoModelForCausalLM
from collections import defaultdict
from src import MODEL_SAVE_PATH

# Model configurations
local_quantized_models = {
    "Llama-3-8B-AWQ-4bit-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-AWQ-4"),
    "Llama-3-8B-BNB-4bit-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-BNB-4"),
    "Llama-3-8B-HQQ-mixed-local": os.path.join(MODEL_SAVE_PATH, "Meta-Llama-3-8B-HQQ-mixed"),
}

def load_model(model_name):
    model_path = local_quantized_models[model_name]
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="cuda")
    return model

def analyze_weight_bits(model):
    bit_counts = defaultdict(int)
    total_params = 0
    
    for name, param in model.named_parameters():
        if param.dtype in [torch.float32, torch.float16]:
            bit_counts['32 or 16'] += param.numel()
        elif hasattr(param, 'quant_state'):
            # AWQ quantized weights
            if hasattr(param.quant_state, 'bits'):
                bits = param.quant_state.bits
            else:
                # If 'bits' attribute is not available, try to infer from the data
                if hasattr(param.quant_state, 'qweight'):
                    qweight = param.quant_state.qweight
                    if qweight.dtype == torch.uint8:
                        max_val = torch.max(qweight).item()
                        if max_val <= 15:
                            bits = 4
                        else:
                            bits = 8
                    else:
                        bits = 'unknown'
                else:
                    bits = 'unknown'
            bit_counts[f'{bits}'] += param.numel()
            print(f"AWQ param: {name}, shape: {param.shape}, bits: {bits}")
        elif hasattr(param, 'qweight'):
            # BNB quantized weights
            if param.qweight.dtype == torch.uint8:
                bit_counts['8'] += param.numel()
            elif param.qweight.dtype == torch.float16:
                max_val = torch.max(param.qweight).item()
                if max_val <= 15:
                    bit_counts['4'] += param.numel()
                else:
                    bit_counts['8'] += param.numel()
        elif param.dtype == torch.int8:
            # HQQ or other int8 quantized weights
            bit_counts['8'] += param.numel()
        else:
            print(f"Unknown quantization for parameter: {name}")
        
        total_params += param.numel()
    
    return bit_counts, total_params

def print_bit_distribution(bit_counts, total_params):
    print("\nBit Distribution:")
    for bits, count in bit_counts.items():
        percentage = (count / total_params) * 100
        print(f"{bits}-bit parameters: {count} ({percentage:.2f}%)")

def main():
    for model_name in local_quantized_models.keys():
        print(f"\nAnalyzing model: {model_name}")
        model = load_model(model_name)
        bit_counts, total_params = analyze_weight_bits(model)
        print_bit_distribution(bit_counts, total_params)
        print(f"Total parameters: {total_params}")

if __name__ == "__main__":
    main()

We suggest you to set `torch_dtype=torch.float16` for better efficiency with AWQ.



Analyzing model: Llama-3-8B-AWQ-4bit-local


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.



Bit Distribution:
32 or 16-bit parameters: 1050939392 (100.00%)
Total parameters: 1050939392

Analyzing model: Llama-3-8B-BNB-4bit-local
AWQ param: model.layers.0.self_attn.q_proj.weight, shape: torch.Size([8388608, 1]), bits: unknown
AWQ param: model.layers.0.self_attn.k_proj.weight, shape: torch.Size([2097152, 1]), bits: unknown
AWQ param: model.layers.0.self_attn.v_proj.weight, shape: torch.Size([2097152, 1]), bits: unknown
AWQ param: model.layers.0.self_attn.o_proj.weight, shape: torch.Size([8388608, 1]), bits: unknown
AWQ param: model.layers.0.mlp.gate_proj.weight, shape: torch.Size([29360128, 1]), bits: unknown
AWQ param: model.layers.0.mlp.up_proj.weight, shape: torch.Size([29360128, 1]), bits: unknown
AWQ param: model.layers.0.mlp.down_proj.weight, shape: torch.Size([29360128, 1]), bits: unknown
AWQ param: model.layers.1.self_attn.q_proj.weight, shape: torch.Size([8388608, 1]), bits: unknown
AWQ param: model.layers.1.self_attn.k_proj.weight, shape: torch.Size([2097152, 1]), bi

ValueError: The model's quantization config from the arguments has no `quant_method` attribute. Make sure that the model has been correctly quantized